# Steam Games EDA
v2.0: added a new colum to our parquet called release_date

Parquet and Requirements Set Up

In [1]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark import StorageLevel
import matplotlib.pyplot as plt
import seaborn as sns
import ast
import json
from datetime import date

Matplotlib created a temporary cache directory at /scratch/jlimfueco/job_48607454/matplotlib-4cdwbxv8 because the default path (/home/jovyan/.cache/matplotlib) is not a writable directory; it is highly recommended to set the MPLCONFIGDIR environment variable to a writable directory, in particular to speed up the import of Matplotlib and to better support multiprocessing.


In [2]:
# INITIALIZING THE SPARK

spark = (
    SparkSession.builder
    .appName("steam_games_set")
    .config("spark.driver.memory", "4g")
    .config("spark.driver.cores", "1")
    .config("spark.executor.instances", "8")
    .config("spark.executor.cores", "1")
    .config("spark.executor.memory", "10g")
    .getOrCreate()
)

In [3]:
# Loading up our parquet
# LOAD THE DATA FROM WHEREVER YOU ARE HOLDING IT!
df = spark.read.parquet('steam_tfidf_nn_recommender_v2.parquet')

df.printSchema()
# The show ends up really really messy here... so eventually we'll make functions and whatnot to make it readable
# View at your own risk
# df.show(5, truncate=False)

root
 |-- appid: long (nullable = true)
 |-- name: string (nullable = true)
 |-- release_date: string (nullable = true)
 |-- price: double (nullable = true)
 |-- genres: string (nullable = true)
 |-- tags: string (nullable = true)
 |-- peak_ccu: long (nullable = true)
 |-- estimated_owners: string (nullable = true)
 |-- average_playtime_forever: long (nullable = true)
 |-- pct_pos_total: long (nullable = true)
 |-- num_reviews_total: long (nullable = true)
 |-- top_50_competitors_json: string (nullable = true)



In [4]:
# Check the Spark details
print('master:', spark.sparkContext.master)
print('partitions:', df.rdd.getNumPartitions())

master: local[*]
partitions: 8


# Let's really quickly go over the columns and what purpose they serve:
## Column Overview Table
| Column                     |               Type | Description                                                                          |
| -------------------------- | -----------------: | ------------------------------------------------------------------------------------ |
| `appid`                    |            Integer | Unique identifier for each game on Steam                                             |
| `name`                     |             String | Name/title of the game                                                               |
| `release_date`             |      String / Date | Official release date of the game, used to evaluate recency and market timing        |
| `price`                    |              Float | Price of the game; `0.0` indicates free-to-play                                      |
| `genres`                   | String / list-like | List of Steam-defined game genres, stored as a string                                |
| `tags`                     | String / dict-like | User-generated tags with associated weights/popularity values                        |
| `peak_ccu`                 |            Integer | Peak concurrent users; maximum number of players online at one time                  |
| `estimated_owners`         |             String | Estimated range of total owners, such as `"10,000,000 - 20,000,000"`                 |
| `average_playtime_forever` |            Integer | Average total playtime per user in minutes                                           |
| `pct_pos_total`            |            Integer | Percentage of positive reviews, from `0–100`                                         |
| `num_reviews_total`        |            Integer | Total number of reviews for the game                                                 |
| `top_50_competitors_json`  | String / JSON-like | Top 50 most similar games based on tag/genre similarity, including similarity scores |


The dataset combines metadata, engagement metrics, and community-driven features. Notably, the tags field provides weighted descriptors derived from user labeling behavior, which serve as the primary feature space for similarity modeling. Additionally, the top_50_competitors_json field contains precomputed nearest neighbors based on similarity metrics, enabling external validation of recommendation models.

(V2.0)

The dataset combines game metadata, release timing, engagement metrics, review metrics, and community-driven content features. The newly added release_date column is especially important because it allows us to separate newer competitors from older legacy titles. This helps make the competition analysis more realistic, since recent releases are more likely to reflect current player attention and market movement. The tags field provides weighted descriptors based on user labeling behavior, while top_50_competitors_json contains precomputed nearest-neighbor competitors that can be analyzed for similarity and recency.

In short, we can partition our understanding of the data to the following:
## Feature Groups
| Category                | Features                                                   | Description                                                           |
| ----------------------- | ---------------------------------------------------------- | --------------------------------------------------------------------- |
| Identifiers             | `appid`, `name`                                            | Unique game identifiers                                               |
| Time / Release Metadata | `release_date`                                             | Used to measure game recency, release cohorts, and competition timing |
| Basic Metadata          | `price`, `genres`                                          | Basic descriptive attributes of each game                             |
| Engagement              | `peak_ccu`, `average_playtime_forever`, `estimated_owners` | Player activity and popularity indicators                             |
| Review Metrics          | `pct_pos_total`, `num_reviews_total`                       | Community sentiment and review volume                                 |
| Content Features        | `tags`                                                     | Weighted tag-based representation of gameplay characteristics         |
| Similarity Data         | `top_50_competitors_json`                                  | Precomputed similar games used for competitor analysis                |



In [5]:
# Here is a fun peak at the mess we're going to clean up...
df.head()

Row(appid=730, name='Counter-Strike 2', release_date='2012-08-21', price=0.0, genres="['Action', 'Free To Play']", tags="{'FPS': 90857, 'Shooter': 65397, 'Multiplayer': 62332, 'Competitive': 53359, 'Action': 47512, 'Team-Based': 46430, 'e-sports': 43533, 'Tactical': 41354, 'First-Person': 39414, 'PvP': 34470, 'Online Co-Op': 33966, 'Co-op': 30263, 'Strategy': 30111, 'Military': 28699, 'War': 28006, 'Difficult': 25966, 'Trading': 25717, 'Realistic': 25430, 'Fast-Paced': 25318, 'Moddable': 6603}", peak_ccu=1212356, estimated_owners='100000000 - 200000000', average_playtime_forever=33189, pct_pos_total=86, num_reviews_total=8632939, top_50_competitors_json='[{"appid": 359550, "name": "Tom Clancy\'s Rainbow Six\\u00ae Siege", "release_date": "2015-12-01", "similarity": 0.7569}, {"appid": 222880, "name": "Insurgency", "release_date": "2014-01-22", "similarity": 0.7523}, {"appid": 1218470, "name": "Lightphobe", "release_date": "2024-02-12", "similarity": 0.6732}, {"appid": 240, "name": "Coun

# So why are we using this data?

Recall that the objective of this project is to analyze competitive relationships between games in the context of player retention—specifically identifying where player attention may shift across similar titles. Rather than building a traditional recommendation system, the goal is to approximate substitution effects (i.e., which games are most likely to draw players away from others) based on shared gameplay characteristics.

## Primary Feature(s)

### tags
The tags field serves as the primary feature for modeling similarity between games. These tags are user-generated descriptors (e.g., FPS, Multiplayer, MOBA) accompanied by weights that reflect their prevalence within each game. Because they capture detailed aspects of gameplay mechanics, pacing, and player experience, tags provide a high-resolution representation of what makes games similar from a player’s perspective.

This is central to the project’s objective: games with highly similar tag profiles are more likely to compete for the same player base. As such, tag-based similarity serves as a proxy for competitive overlap in player interest.

### genres
The genres field provides a broader categorization of each game (e.g., Action, Strategy). While less granular than tags, genres help reinforce high-level similarities and provide context for grouping games into broader competitive categories.

For example, two games within the same genre but with different tag profiles may compete differently across player segments. 

## Supporting Feature(s)

### top_50_competitors_json
This field contains precomputed lists of similar games along with similarity scores. It is not used as an input to the model in order to avoid circular reasoning or data leakage. Instead, it serves as a reference benchmark to evaluate whether the similarity structure derived from tag-based modeling aligns with an external notion of competition.

This enables validation of whether the model is capturing meaningful competitive relationships between games.

## Contextual Features (Not Used for Similarity Modeling)

The following features are retained for exploratory analysis and interpretation but are not used to compute similarity:

### price
While price may influence purchasing decisions, it does not directly reflect gameplay similarity or player experience.

### peak_ccu, estimated_owners
These features measure popularity and scale. They are useful for identifying dominant titles or market concentration but do not define whether two games compete for the same type of player.

### average_playtime_forever
This metric provides insight into player engagement and retention but does not characterize the nature of gameplay itself.

### pct_pos_total, num_reviews_total
These features capture user sentiment and review volume. While valuable for understanding game reception, they do not directly contribute to modeling competitive similarity.

### release_date
This feature provides temporal context by indicating when a game entered the market. It is valuable for distinguishing newer active competitors from older legacy titles, evaluating lifecycle stage, and improving interpretation of engagement metrics. While strategically important for analysis, release timing does not directly measure gameplay similarity or player preference structure.

In [6]:
# NOTE
# If you use our parquet for your own purposes, this code block will make the data a lot cleaner for use.

# REALLY QUICKLY:
# Let's convert messy, string-encoded JSON columns into structured,
# queryable Spark data types for EDA and modeling.

# --- Convert release_date into Spark date type ---
df = df.withColumn('release_dt', F.to_date(F.col('release_date')))

# --- Fix quotes first (since JSON-like columns use single quotes) ---
df = df.withColumn('tags_fixed', F.regexp_replace(F.col('tags'), "'", '"'))
df = df.withColumn('genres_fixed', F.regexp_replace(F.col('genres'), "'", '"'))
df = df.withColumn('competitors_fixed', F.regexp_replace(F.col('top_50_competitors_json'), "'", '"'))

# --- Define schemas ---
tags_schema = MapType(StringType(), IntegerType())
genres_schema = ArrayType(StringType())

competitor_schema = ArrayType(
    StructType([
        StructField('appid', IntegerType()),
        StructField('name', StringType()),
        StructField('similarity', DoubleType())
    ])
)

# --- Parse into proper structures ---
df = df.withColumn('tags_map', F.from_json(F.col('tags_fixed'), tags_schema))
df = df.withColumn('genres_array', F.from_json(F.col('genres_fixed'), genres_schema))
df = df.withColumn('competitors', F.from_json(F.col('competitors_fixed'), competitor_schema))

# --- Drop messy helper columns, but keep original release_date and parsed release_dt ---
df_clean = df.drop(
    'tags', 'genres', 'top_50_competitors_json',
    'tags_fixed', 'genres_fixed', 'competitors_fixed'
)

df_clean.cache()
df_clean.limit(1).collect()

[Row(appid=730, name='Counter-Strike 2', release_date='2012-08-21', price=0.0, peak_ccu=1212356, estimated_owners='100000000 - 200000000', average_playtime_forever=33189, pct_pos_total=86, num_reviews_total=8632939, release_dt=datetime.date(2012, 8, 21), tags_map={'Shooter': 65397, 'Tactical': 41354, 'Co-op': 30263, 'Realistic': 25430, 'Action': 47512, 'FPS': 90857, 'War': 28006, 'PvP': 34470, 'Online Co-Op': 33966, 'e-sports': 43533, 'First-Person': 39414, 'Team-Based': 46430, 'Competitive': 53359, 'Multiplayer': 62332, 'Difficult': 25966, 'Trading': 25717, 'Strategy': 30111, 'Fast-Paced': 25318, 'Moddable': 6603, 'Military': 28699}, genres_array=['Action', 'Free To Play'], competitors=None)]

In [7]:
# This is a nifty developed helper fxn to let us read into particular games that exist
# top_n_comp can only go to 50

from datetime import date

def inspect_game(df_clean, game_name, raw_df=None, top_n_tags=10, top_n_comp=10, sort_comp_by='similarity'):
    row = (
        df_clean.filter(F.col('name') == game_name)
                .limit(1)
                .collect()
    )

    if not row:
        print(f'Game not found: {game_name}')
        return

    r = row[0].asDict()

    genres = r.get('genres_array', [])
    tags = dict(r.get('tags_map', {}))
    competitors = r.get('competitors', [])

    if (not competitors) and raw_df is not None:
        raw_row = (
            raw_df.filter(F.col('name') == game_name)
                  .select('top_50_competitors_json')
                  .limit(1)
                  .collect()
        )

        if raw_row and raw_row[0]['top_50_competitors_json'] is not None:
            try:
                competitors = json.loads(raw_row[0]['top_50_competitors_json'])
            except Exception as e:
                print(f'Competitor parse failed: {e}')
                competitors = []

    print('\nGAME INFO')
    print(f"Name: {r.get('name')}")
    print(f"AppID: {r.get('appid')}")
    print(f"Release Date: {r.get('release_date')}")
    print(f"Price: {r.get('price')}")
    print(f"Peak CCU: {r.get('peak_ccu')}")
    print(f"Estimated Owners: {r.get('estimated_owners')}")
    print(f"Avg Playtime: {r.get('average_playtime_forever')}")
    print(f"Pct Positive: {r.get('pct_pos_total')}")
    print(f"Total Reviews: {r.get('num_reviews_total')}")

    print('\nGENRES')
    print(genres)

    print('\nTOP TAGS')
    if tags:
        top_tags = sorted(tags.items(), key=lambda x: -x[1])[:top_n_tags]

        for tag, val in top_tags:
            print(f'{tag}: {val}')
    else:
        print(tags)

    print(f'\nTOP COMPETITORS sorted by {sort_comp_by}')

    if not competitors:
        print([])
        return

    comp_appids = [
        int(comp.get('appid'))
        for comp in competitors
        if comp.get('appid') is not None
    ]

    comp_lookup = (
        df_clean
        .filter(F.col('appid').isin(comp_appids))
        .select('appid', 'release_date', 'release_dt')
        .collect()
    )

    comp_dates = {
        row['appid']: {
            'release_date': row['release_date'],
            'release_dt': row['release_dt']
        }
        for row in comp_lookup
    }

    enriched_competitors = []

    for comp in competitors:
        appid = int(comp.get('appid'))
        date_info = comp_dates.get(appid, {})

        enriched_competitors.append({
            'name': comp.get('name'),
            'appid': appid,
            'similarity': comp.get('similarity', 0),
            'release_date': date_info.get('release_date'),
            'release_dt': date_info.get('release_dt')
        })

    if sort_comp_by == 'release_date':
        enriched_competitors = sorted(
            enriched_competitors,
            key=lambda x: x['release_dt'] or date(1900, 1, 1),
            reverse=True
        )

    elif sort_comp_by == 'hybrid':
        enriched_competitors = sorted(
            enriched_competitors,
            key=lambda x: (
                x['release_dt'] or date(1900, 1, 1),
                x['similarity'] or 0
            ),
            reverse=True
        )

    else:
        enriched_competitors = sorted(
            enriched_competitors,
            key=lambda x: x['similarity'] or 0,
            reverse=True
        )

    for comp in enriched_competitors[:top_n_comp]:
        print(
            f"{comp.get('name')} "
            f"(appid={comp.get('appid')}, "
            f"release={comp.get('release_date')}, "
            f"sim={round(comp.get('similarity', 0), 4)})"
        )

# Let's talk about some games in the Steam catalog:
| Game   | Type | Core Loop                                  |
| ------ | ---- | ------------------------------------------ |
| CS2    | FPS  | Mechanical skill, short rounds             |
| Dota 2 | MOBA | Strategy, long sessions, team coordination |


## Why this function and view of the data matters:

We're moving a bit far ahead from the machine learning question of why a player didn't return to play, but here is how we can investigate the external factors regarding that decision. Presume the game a player didn't return to, we can use our game inspector to see non-graphical information.

Instead of a recommender system, consider the reverse side of the story: a user sees recommendations as other games for them to explore. A business sees these other games as competition. Now we're looking at this info as to why someone is moving away from one product to another.

In [8]:
# Let's use our helper!

# Traditional nearest competitors (gameplay similarity)
inspect_game(
    df_clean,
    'Counter-Strike 2',
    raw_df=df,
    sort_comp_by='similarity'
)

# Most recent similar competitors (market freshness)
inspect_game(
    df_clean,
    'Counter-Strike 2',
    raw_df=df,
    sort_comp_by='release_date'
)

# Which recent games are also meaningfully similar and therefore realistic current competitors?
inspect_game(
    df_clean,
    'Counter-Strike 2',
    raw_df=df,
    sort_comp_by='hybrid'
)


GAME INFO
Name: Counter-Strike 2
AppID: 730
Release Date: 2012-08-21
Price: 0.0
Peak CCU: 1212356
Estimated Owners: 100000000 - 200000000
Avg Playtime: 33189
Pct Positive: 86
Total Reviews: 8632939

GENRES
['Action', 'Free To Play']

TOP TAGS
FPS: 90857
Shooter: 65397
Multiplayer: 62332
Competitive: 53359
Action: 47512
Team-Based: 46430
e-sports: 43533
Tactical: 41354
First-Person: 39414
PvP: 34470

TOP COMPETITORS sorted by similarity
Tom Clancy's Rainbow Six® Siege (appid=359550, release=2015-12-01, sim=0.7569)
Insurgency (appid=222880, release=2014-01-22, sim=0.7523)
Lightphobe (appid=1218470, release=2024-02-12, sim=0.6732)
Counter-Strike: Source (appid=240, release=2004-11-01, sim=0.6094)
BATTALION: Legacy (appid=489940, release=2019-05-23, sim=0.607)
Splitgate (appid=677620, release=2019-05-24, sim=0.6004)
Ravenfield (appid=636480, release=2017-05-18, sim=0.5946)
Aim Hero (appid=518030, release=2016-09-05, sim=0.5766)
Warfare : Battleground™ (appid=2890270, release=2024-04-30, s

### Similarity Sort

This view ranks competitors strictly by similarity score, highlighting games that most closely match the target title in terms of tags, genres, and gameplay characteristics. It is the best representation of structural competition, showing which games appeal to similar player preferences regardless of release timing or market age.

### Release Date Sort

This view ranks competitors by most recent release date, emphasizing newer titles that may represent current or emerging competition. While these games may vary in similarity strength, this perspective is useful for understanding recent market movement and identifying where player attention may shift today.

### Hybrid Sort

This view combines release recency with similarity, prioritizing newer competitors while still favoring games with stronger gameplay overlap. It offers a practical business-oriented ranking by identifying titles that are both currently relevant and realistically capable of competing for the same player base.

## Using `inspect_game` for Exploratory Analysis

The `inspect_game` helper provides a structured snapshot of an individual title, allowing us to quickly examine multiple dimensions of a game in one place.

With this function, we can:

- Analyze **genre composition** to understand how a game is categorized at a high level  
- Explore **top tags** to capture more granular gameplay characteristics and player-facing features  
- Identify **competitor relationships** through multiple ranking methods:
  - **Similarity Sort** → closest gameplay substitutes  
  - **Release Date Sort** → newest relevant competitors  
  - **Hybrid Sort** → recent competitors with strong similarity overlap  
- Compare **engagement and popularity metrics** (e.g., playtime, reviews, CCU) across different titles  
- Evaluate how **release timing** may influence current competitive relevance

This enables rapid, game-level comparisons and helps build intuition about how different games are positioned within the broader ecosystem. With the addition of release dates, the helper now supports both static similarity analysis and dynamic market competition views.

## Counter-Strike 2 — Observations

### Scale & Popularity

- Counter-Strike 2 is one of the most dominant games in the dataset, with a **peak CCU of ~1.21M players**.
- It has an estimated **100M–200M owners** and over **8.6M total reviews**, indicating exceptional global reach and long-term market relevance.
- Despite being a long-established franchise, CS2 remains one of the most actively played titles today.

### Player Engagement

- The **average playtime of 33,189 minutes (~553 hours)** reflects **strong long-term engagement**.
- This suggests players repeatedly return despite the game’s relatively short match structure, highlighting strong replayability and competitive retention.
- The title demonstrates how session-based gameplay can still generate deep lifetime engagement.

### Player Sentiment

- With **86% positive reviews**, CS2 demonstrates **high player satisfaction at scale**.
- Sustaining this level of sentiment across millions of reviews indicates a mature and trusted multiplayer ecosystem.
- Positive reception at this scale often reinforces long-term retention through network effects and community confidence.

### Genre Identity

- CS2 is categorized under **Action** and **Free-to-Play**, placing it firmly within the **fast-paced multiplayer FPS space**.
- Its streamlined genre labeling reflects a focused competitive identity rather than a hybrid design structure.
- The free-to-play model likely lowers entry barriers and supports continued player acquisition.

### Tag Profile (Gameplay Characteristics)

- Dominant tags include **FPS, Shooter, Multiplayer, Competitive, Team-Based, and Tactical**, emphasizing:

  - **mechanical skill and precision** (FPS, Shooter)
  - **structured coordination** (Team-Based)
  - **competitive intensity** (Competitive, PvP, e-sports)

- These tags define a **fast-paced, reaction-driven gameplay loop** built around mastery and repeat competition.
- The profile strongly suggests players are retained through skill improvement, rank progression, and team-based mastery.

### Competitive Ecosystem

- Top competitors include titles such as **Rainbow Six Siege**, **Insurgency**, and newer FPS entries depending on sort method.
- Similarity rankings identify structural substitutes, while release-based rankings surface more recent competitive threats.
- This demonstrates that competition exists across both established legacy shooters and newer market entrants.

### Key Takeaways

- Counter-Strike 2 represents a **mechanically driven, fast-paced competitive experience** with exceptional replayability.
- Its high engagement and sentiment indicate a **stable, long-standing player base**.
- The addition of release-date analysis shows that even older flagship titles must still compete with newer shooters entering the market.

**Conclusion:** CS2 exemplifies how skill-based gameplay, competitive structure, and strong community trust can sustain large-scale player engagement over time.

Counter-Strike 2 also highlights the importance of temporal context. While CS2 and CS:GO represent the same franchise lineage, their identities differ across time. The inclusion of release dates helps distinguish relaunches, platform transitions, and evolving competitive relevance.

This is a really strong place to start looking at games, even for the not so avid gamer. Counter Strike 2 has a long history of players because this isn't the original game: CS2 replaced Counter Strike: Global Offensive

In [9]:
inspect_game(df_clean, 'Counter-Strike: Global Offensive', raw_df=df)

Game not found: Counter-Strike: Global Offensive


Notice here we do not have a direct entry for Counter-Strike: Global Offensive because CS2 effectively replaced CS:GO within the Steam ecosystem. This highlights an important challenge in longitudinal platform data: game identities can evolve through relaunches, renaming, mergers, or product transitions rather than appearing as separate static entries.

CS2 is only one example of a broader market reality. Many major franchises undergo sequels, remasters, definitive editions, platform migrations, or rebranded live-service transitions that complicate clean historical tracking.

It is also important to note that multiple titles may share related names while carrying different `appid` values. These identifiers help distinguish editions, bundles, regional variants, test servers, or re-releases. Preserving `appid` as a unique identifier is therefore essential for accurate analysis.

### Why This Matters for Our Project

Earlier versions of this dataset were more limited in temporal analysis. With the addition of `release_date`, we can now better study:

- franchise transitions over time  
- legacy titles versus modern successors  
- relaunches and evolving market relevance  
- how competition changes across release generations  

This reinforces that the video game market is dynamic rather than static. Similarity between games matters, but timing, lifecycle stage, and title evolution also shape where players migrate.



In [10]:
# Moving on to Dota 2!

# Traditional nearest competitors (gameplay similarity)
inspect_game(df_clean,'Dota 2',raw_df=df,sort_comp_by='similarity')

# Most recent similar competitors (market freshness)
inspect_game(df_clean,'Dota 2',raw_df=df,sort_comp_by='release_date')

# Which recent games are also meaningfully similar and therefore realistic current competitors?
inspect_game(df_clean,'Dota 2',raw_df=df,sort_comp_by='hybrid')


GAME INFO
Name: Dota 2
AppID: 570
Release Date: 2013-07-09
Price: 0.0
Peak CCU: 555977
Estimated Owners: 200000000 - 500000000
Avg Playtime: 43031
Pct Positive: 81
Total Reviews: 2452595

GENRES
['Action', 'Strategy', 'Free To Play']

TOP TAGS
Free to Play: 59933
MOBA: 20158
Multiplayer: 15359
Strategy: 14252
e-sports: 11780
Team-Based: 10962
Competitive: 8286
Action: 7920
Online Co-Op: 7464
PvP: 6046

TOP COMPETITORS sorted by similarity
Dropzone (appid=572520, release=2017-02-15, sim=0.7327)
Strife® (appid=339280, release=2015-05-22, sim=0.634)
Rise of Legions (appid=748940, release=2019-08-30, sim=0.6235)
Clash: Mutants Vs Pirates (appid=822710, release=2024-05-02, sim=0.622)
Isekai Eternal Alpha (appid=1467410, release=2021-07-14, sim=0.6064)
THE DAY Online (appid=795580, release=2018-04-03, sim=0.5983)
Coregrounds (appid=649770, release=2018-04-30, sim=0.5916)
Heat and Run (appid=1225790, release=2022-10-13, sim=0.5755)
Zeal (appid=703280, release=2018-09-24, sim=0.564)
SMITE® (a

## Dota 2 — Observations

### Scale & Popularity

- Dota 2 remains one of the largest multiplayer titles in the dataset, with a **peak CCU of ~556K players**.
- It has an estimated **20M–50M owners** and over **2.45M total reviews**, confirming long-term global relevance.
- Although smaller than Counter-Strike 2 in raw scale, it still represents an elite-tier live service title with a durable player ecosystem.

### Player Engagement

- The **average playtime of 43,031 minutes (~717 hours)** indicates **extremely high long-term engagement**.
- This exceeds Counter-Strike 2 in average playtime, suggesting a player base with deep commitment and repeated long-session behavior.
- MOBA systems often reward mastery, progression, and strategic depth, which can drive strong retention over time.

### Player Sentiment

- With **81% positive reviews**, Dota 2 shows strong user satisfaction across millions of reviews.
- Slightly lower sentiment than CS2 may reflect higher complexity, steeper learning curves, or frustration associated with competitive team environments.
- Even so, sustaining positive sentiment at this scale signals a healthy long-term product.

### Genre Identity

- Dota 2 spans **Action, Strategy, and Free-to-Play**, making it a more hybrid title than CS2.
- The inclusion of **Strategy** distinguishes it from pure shooters and highlights macro-level decision making.
- Its free-to-play model also supports continued player acquisition and ecosystem longevity.

### Tag Profile (Gameplay Characteristics)

- Dominant tags include **Free to Play, MOBA, Multiplayer, Strategy, e-sports, Team-Based, and Competitive**, emphasizing:
    - **strategic coordination** (MOBA, Strategy)
    - **team dependency** (Team-Based, Multiplayer)
    - **competitive mastery** (Ranked, e-sports, PvP)
- Compared with CS2’s reaction-based FPS profile, Dota 2 emphasizes planning, drafting, teamwork, and game knowledge.
- These traits often create highly loyal but harder-to-acquire player communities.

### Competitive Ecosystem

- Similarity rankings surface titles such as **SMITE**, **Dropzone**, and strategy-oriented multiplayer games.
- Release-date rankings reveal newer action-strategy competitors and hybrid online titles entering adjacent market space.
- This suggests Dota 2 competes not only with traditional MOBAs, but with broader team-based strategic multiplayer experiences.

### Key Takeaways

- Dota 2 represents a **deep, strategy-heavy competitive experience** with exceptional retention.
- Its player base appears highly committed, as shown by one of the strongest playtime metrics in the dataset.
- Competition for Dota 2 may come less from direct clones and more from adjacent multiplayer strategy ecosystems.

**Conclusion:** Dota 2 demonstrates how complexity, teamwork, and long-term mastery can sustain a powerful live-service ecosystem even in a highly competitive market.

Dota 2 also reinforces the value of release-date analysis. Legacy flagship titles may remain dominant for years, but newer entrants can still emerge as substitution threats depending on player preferences and market trends.

In [11]:
# Feel free to explore games that you have in your own library!

inspect_game(
    df_clean,
    game_name='INSERT GAME NAME HERE',
    raw_df=df,
    top_n_tags=10,
    top_n_comp=10,   # max = 50
    sort_comp_by='similarity'   # options: similarity, release_date, hybrid
)

Game not found: INSERT GAME NAME HERE


# Co-occurrence analysis (what appears together with what):
### Co-Occurrence Analysis — Motivation & Use Case

Co-occurrence analysis is used to identify how often certain genres or tags appear together within the same game. Rather than looking at features in isolation, this approach helps uncover **structural relationships** between game characteristics.

From a business perspective, this provides insight into:
- **Game design patterns** — which combinations of genres or features are commonly bundled together
- **Market positioning** — how games are grouped within the broader ecosystem
- **Player experience expectations** — what features players are likely to encounter together

For example, if “Action” frequently co-occurs with “Adventure” and “Multiplayer,” this suggests a common design archetype that developers may follow and players may expect.

These relationships can later support more advanced tasks such as:
- identifying **competitive clusters**
- improving **recommendation systems**
- understanding **player migration between similar games**

Overall, co-occurrence analysis moves beyond simple frequency counts to reveal how different elements of games are interconnected.

In [12]:
# helper functions

def genre_co_occurrence(df, target_genre, remove_self=True):
    # filter games containing the target genre
    subset = df.filter(
        F.array_contains(F.col('genres_array'), target_genre)
    )
    
    # explode genres
    co_genres = subset.selectExpr('explode(genres_array) as genre')
    
    # count co-occurrence
    counts = (
        co_genres
        .groupBy('genre')
        .count()
        .orderBy('count', ascending=False)
    )
    
    # optionally remove the genre itself
    if remove_self:
        counts = counts.filter(F.col('genre') != target_genre)
    
    return counts

def tag_co_occurrence(df, target_tag, remove_self=True):
    # explode tags first
    tags_exploded = df.selectExpr('explode(tags_map) as (tag, weight)')
    
    # find all game IDs that contain the target tag
    games_with_tag = df.selectExpr('explode(tags_map) as (tag, weight)', 'appid') \
        .filter(F.col('tag') == target_tag) \
        .select('appid') \
        .distinct()
    
    # join back to get all tags for those games
    subset = df.join(games_with_tag, on='appid', how='inner')
    
    # explode again for co-occurrence
    co_tags = subset.selectExpr('explode(tags_map) as (tag, weight)')
    
    # count co-occurrence
    counts = (
        co_tags
        .groupBy('tag')
        .count()
        .orderBy('count', ascending=False)
    )
    
    # optionally remove itself
    if remove_self:
        counts = counts.filter(F.col('tag') != target_tag)
    
    return counts

def show_genre_co_pd(df, genre, n=10):
    print(f"\n Genres most commonly found with '{genre}':\n")
    display(genre_co_occurrence(df, genre).limit(n).toPandas())


def show_tag_co_pd(df, tag, n=15):
    print(f"\n Tags most commonly found with '{tag}':\n")
    display(tag_co_occurrence(df, tag).limit(n).toPandas())

In [13]:
# Example use case:
# genres related to Action
show_genre_co_pd(df_clean, 'Action')
show_genre_co_pd(df_clean, 'Strategy')
show_genre_co_pd(df_clean, 'Indie')


 Genres most commonly found with 'Action':



,genre,count
0,Indie,27985
1,Adventure,16600
2,Casual,12249
3,RPG,6582
4,Simulation,5325
5,Strategy,5002
6,Early Access,4856
7,Free To Play,3611
8,Sports,1921
9,Racing,1728



 Genres most commonly found with 'Strategy':



,genre,count
0,Indie,12350
1,Casual,8035
2,Simulation,5684
3,Adventure,5176
4,Action,5002
5,RPG,4396
6,Early Access,2223
7,Free To Play,1801
8,Sports,850
9,Massively Multiplayer,707



 Genres most commonly found with 'Indie':



,genre,count
0,Casual,29530
1,Action,27985
2,Adventure,26932
3,Simulation,13005
4,Strategy,12350
5,RPG,11813
6,Early Access,6941
7,Free To Play,5888
8,Sports,2542
9,Racing,2269


### Genre Co-Occurrence Insights

#### Action
- Action games most frequently co-occur with **Indie, Adventure, and Casual**, indicating that Action is often combined with **accessible and broadly appealing gameplay styles**.
- The presence of **RPG, Simulation, and Strategy** suggests that Action is commonly integrated into more complex or hybrid gameplay systems.
- This highlights Action as a **flexible, cross-genre component** that appears across a wide range of game designs rather than forming a narrow category.

---

#### Indie
- Indie games most frequently co-occur with **Casual, Action, and Adventure**, indicating that Indie titles are often built around **accessible and broadly appealing gameplay styles**.
- The presence of **Simulation, Strategy, and RPG** suggests that Indie is not a genre itself, but rather a **development classification that spans multiple gameplay types**.
- This reinforces that Indie games serve as a **cross-cutting category**, rather than a distinct gameplay ecosystem.

---

#### Strategy
- Strategy games most commonly co-occur with **Indie, Casual, and Simulation**, indicating that many strategy titles are developed in **lower-budget or niche design spaces**.
- The strong link with **Simulation** highlights a shared emphasis on **systems-based and decision-driven gameplay**.
- Compared to Indie, Strategy appears more **structurally focused**, with tighter relationships to specific gameplay mechanics rather than broad appeal.

---

### Key Takeaway(s)
- Genre relationships are not isolated; instead, they form **overlapping clusters of gameplay styles**.
- **Indie acts as a broad umbrella**, while genres like **Strategy form tighter, mechanic-driven groupings**.
- This suggests that analyzing games by a single genre may be insufficient, as most titles exist within **multi-genre ecosystems**.

In [14]:
# tags related to Multiplayer
show_tag_co_pd(df_clean, 'Multiplayer')


 Tags most commonly found with 'Multiplayer':



,tag,count
0,Singleplayer,5090
1,Action,4991
2,Indie,4153
3,Co-op,3151
4,PvP,2999
5,Casual,2987
6,Adventure,2548
7,Strategy,2387
8,Online Co-Op,2220
9,3D,2210


In [15]:
# tags related to Indie
show_tag_co_pd(df_clean, 'Indie')


 Tags most commonly found with 'Indie':



,tag,count
0,Singleplayer,24274
1,Casual,19764
2,Adventure,17988
3,Action,17403
4,2D,12016
5,Puzzle,8705
6,Strategy,8653
7,Simulation,8553
8,RPG,7386
9,Atmospheric,7095


## Tag Co-Occurrence Insights

### Multiplayer

- The strongest co-occurring tags with **Multiplayer** include **Singleplayer, Action, Indie, Co-op, PvP, and Strategy**, showing that multiplayer is often paired with a wide variety of game structures rather than existing as a standalone identity.
- The presence of **Singleplayer** suggests many games support both solo and online modes, reflecting flexible player-access models.
- Tags such as **Co-op, PvP, Online Co-Op, and Local Multiplayer** indicate that multiplayer experiences are highly diverse, spanning cooperative, competitive, and social play formats.
- The appearance of **First-Person** and **3D** suggests that many multiplayer titles lean toward immersive or action-oriented gameplay environments.

### Indie

- The strongest co-occurring tags with **Indie** include **Singleplayer, Casual, Adventure, Action, and 2D**, indicating that indie games are heavily concentrated in accessible, lower-barrier play experiences.
- The prominence of **Puzzle, Atmospheric, Pixel Graphics, Story Rich, Cute, and Colorful** suggests indie titles frequently emphasize creativity, narrative identity, and artistic presentation.
- Strategy, Simulation, and RPG also appear strongly, showing that indie development spans both casual and mechanically deep genres.
- Compared with Multiplayer, the Indie tag appears more associated with design style and production identity than with a single gameplay mode.

### Key Takeaway(s)

- **Multiplayer** functions as a broad interaction format that cuts across many genres and gameplay styles.
- **Indie** behaves more like a development ecosystem, associated with creative experimentation, niche experiences, and varied mechanics.
- Tags often represent different dimensions of a game (mode, style, mechanics, identity), so analyzing co-occurrence helps reveal how player experiences are layered rather than defined by one label alone.

# “What are all the possible genres and tags in this dataset?”
In case you wanted to do your own exploring--

show_genre_co_pd(df_clean, 'INSERT GENRE')

OR

show_tag_co_pd(df_clean, 'INSERT TAG')

In [16]:
# get all unique genres
all_genres = (
    df_clean
    .selectExpr('explode(genres_array) as genre')
    .distinct()
    .orderBy('genre')
)

# display
display(all_genres.toPandas())

,genre
0,360 Video
1,Accounting
2,Action
3,Adventure
4,Animation & Modeling
5,Audio Production
6,Casual
7,Design & Illustration
8,Documentary
9,Early Access


In [17]:
# get all unique tags
all_tags = (
    df_clean
    .selectExpr('explode(tags_map) as (tag, weight)')
    .select('tag')
    .distinct()
    .orderBy('tag')
)

# display
display(all_tags.toPandas())

,tag
0,1980s
1,2.5D
2,2D
3,2D Fighter
4,2D Platformer
...,...
444,World War I
445,World War II
446,Wrestling
447,Zombies


# Data Quality Check
We've been doing a lot of surface level diving but we need to Quality Check the Data.

## Missingness

In [21]:
# Missingness check on original/core analysis columns only
# release_dt is the cleaned/parsed version of release_date, so release_date is excluded

target_df = df_clean.select(
    'appid',
    'name',
    'price',
    'peak_ccu',
    'estimated_owners',
    'average_playtime_forever',
    'pct_pos_total',
    'num_reviews_total',
    'release_dt'
)

exprs = []

for col_name, dtype in target_df.dtypes:
    cond = F.col(col_name).isNull()
    
    if dtype in ['double', 'float']:
        cond = cond | F.isnan(F.col(col_name))
    
    if dtype == 'string':
        cond = cond | (F.trim(F.col(col_name)) == '')
    
    exprs.append(F.count(F.when(cond, col_name)).alias(col_name))

missing_summary = target_df.select(exprs)

missing_summary.show(vertical=True, truncate=False)

-RECORD 0-----------------------
 appid                    | 0   
 name                     | 2   
 price                    | 0   
 peak_ccu                 | 0   
 estimated_owners         | 0   
 average_playtime_forever | 0   
 pct_pos_total            | 0   
 num_reviews_total        | 0   
 release_dt               | 0   



Missingness was checked across the core analysis columns after excluding engineered fields. Overall, the dataset is highly complete: key variables such as price, peak concurrent users, estimated owners, average playtime, review metrics, and parsed release dates show no missing values. The only missingness found was in the `name` column, with 2 records missing game titles. Since this is a very small number of rows, these records can be inspected directly rather than requiring major preprocessing.

In [22]:
# Inspect records where game name is missing

missing_names = (
    df_clean
    .filter(
        F.col('name').isNull() |
        (F.trim(F.col('name')) == '')
    )
    .select(
        'appid',
        'name',
        'release_dt',
        'price',
        'peak_ccu',
        'estimated_owners',
        'num_reviews_total'
    )
)

missing_names.show(truncate=False)

+-------+----+----------+-----+--------+----------------+-----------------+
|appid  |name|release_dt|price|peak_ccu|estimated_owners|num_reviews_total|
+-------+----+----------+-----+--------+----------------+-----------------+
|396420 |NULL|2016-11-01|0.0  |0       |0 - 0           |65               |
|1116910|NULL|2019-09-25|6.99 |0       |0 - 0           |30               |
+-------+----+----------+-----+--------+----------------+-----------------+



## Duplicates and Justification

In [34]:
# ==========================================
# Duplicate Game Name Explorer
# Lists every game record where the name maps to multiple appids
# ==========================================

duplicate_summary = (
    df_clean
    .groupBy('name')
    .agg(
        F.countDistinct('appid').alias('num_appids')
    )
    .filter(
        (F.col('name').isNotNull()) &
        (F.trim(F.col('name')) != '') &
        (F.col('num_appids') > 1)
    )
)

duplicate_explorer = (
    df_clean
    .join(duplicate_summary, on='name', how='inner')
    .select(
        'name',
        'appid',
        'num_appids',
        'release_dt',
        'price',
        'peak_ccu',
        'estimated_owners',
        'average_playtime_forever',
        'pct_pos_total',
        'num_reviews_total'
    )
    .orderBy(
        F.desc('num_appids'),
        F.asc('name'),
        F.asc('appid')
    )
)

#duplicate_explorer.show(100, truncate=False)

In [35]:
# Cleaner duplicate-name dataframe
# Shows only title + number of distinct appids

duplicate_summary = (
    df_clean
    .groupBy('name')
    .agg(
        F.countDistinct('appid').alias('num_appids')
    )
    .filter(
        (F.col('name').isNotNull()) &
        (F.trim(F.col('name')) != '') &
        (F.col('num_appids') > 1)
    )
    .orderBy(
        F.desc('num_appids'),
        F.asc('name')
    )
)

duplicate_summary.show(25, truncate=False)

+---------------------------------------------+----------+
|name                                         |num_appids|
+---------------------------------------------+----------+
|Shadow of the Tomb Raider: Definitive Edition|20        |
|Alone                                        |6         |
|Aurora                                       |5         |
|Echoes                                       |5         |
|Tom Clancy's Rainbow Six® Siege              |5         |
|Arena                                        |4         |
|Bounce                                       |4         |
|Delirium                                     |4         |
|EA SPORTS FC™ 24                             |4         |
|EA SPORTS FC™ 25                             |4         |
|Escape                                       |4         |
|Exodus                                       |4         |
|Hide and Seek                                |4         |
|Loop                                         |4        

In [37]:
# ==========================================
# Search Duplicate Versions by Game Name
# Change the text below to inspect any title
# ==========================================

game_search = 'Shadow of the Tomb Raider: Definitive Edition'

duplicate_lookup = (
    df_clean
    .join(
        duplicate_summary.select('name'),
        on='name',
        how='inner'
    )
    .filter(F.col('name') == game_search)
    .select(
        'name',
        'appid',
        'release_dt',
        'price',
        'peak_ccu',
        'estimated_owners',
        'average_playtime_forever',
        'pct_pos_total',
        'num_reviews_total'
    )
    .orderBy('release_dt', 'appid')
)

# duplicate_lookup.show(truncate=False)

To support manual review, a search tool was created to inspect titles associated with multiple Steam app IDs. This allows readers to explore whether repeated names reflect remasters, editions, bundles, demos, or unrelated games sharing the same title.

In [39]:
# FOR YOU TO EXPLORE!
# Put a game in the game search--> works partially too
game_search = 'Tomb Raider'

duplicate_lookup = (
    df_clean
    .join(
        duplicate_summary.select('name'),
        on='name',
        how='inner'
    )
    .filter(
        F.lower(F.col('name')).contains(game_search.lower())
    )
    .select(
        'name',
        'appid',
        'release_dt',
        'price',
        'peak_ccu',
        'estimated_owners',
        'num_reviews_total'
    )
    .orderBy('name', 'release_dt')
)

duplicate_lookup.show(30, truncate=False)

+---------------------------------------------+------+----------+-----+--------+-----------------+-----------------+
|name                                         |appid |release_dt|price|peak_ccu|estimated_owners |num_reviews_total|
+---------------------------------------------+------+----------+-----+--------+-----------------+-----------------+
|Shadow of the Tomb Raider: Definitive Edition|750920|2018-09-14|0.0  |980     |2000000 - 5000000|61638            |
|Shadow of the Tomb Raider: Definitive Edition|905320|2018-09-14|0.0  |0       |0 - 20000        |61629            |
|Shadow of the Tomb Raider: Definitive Edition|890031|2018-09-14|0.0  |0       |0 - 20000        |61629            |
|Shadow of the Tomb Raider: Definitive Edition|890032|2018-09-14|0.0  |0       |0 - 20000        |61629            |
|Shadow of the Tomb Raider: Definitive Edition|890030|2018-09-14|0.0  |0       |0 - 20000        |61629            |
|Shadow of the Tomb Raider: Definitive Edition|849167|2018-09-14

The duplicate title search for *Shadow of the Tomb Raider: Definitive Edition* illustrates why repeated game names should not automatically be treated as data errors. Although the same title appears under many different `appid` values, the records share identical release dates and highly similar pricing/review patterns, suggesting these entries likely represent alternate Steam store listings such as bundles, editions, packages, regional records, or platform-specific versions rather than separate games. This supports retaining `appid` as the true unique identifier while treating duplicate names as expected marketplace behavior.

In [31]:
print('Number of duplicated game names:', duplicate_summary.count())
print('Number of appid records affected:', duplicate_versions.count())

Number of duplicated game names: 639
Number of appid records affected: 1394


Duplicate title analysis found that some game names appear under multiple Steam `appid` values. These are not necessarily errors because Steam often separates different versions, editions, demos, or legacy releases into unique app IDs. Since `appid` is the true unique identifier, these records were retained, but duplicate names were documented as a possible source of ambiguity when interpreting game-level results.

### Summary of Data Quality Assurance Checks

A final round of data quality checks was conducted to evaluate missingness, duplicate identities, and general consistency across the dataset. Missing values were minimal across the core analytical variables, with only two records lacking game titles and no notable gaps in price, ownership, playtime, review, or release-date fields. Duplicate title analysis showed that some names map to multiple Steam `appid` values, which is common for editions, bundles, demos, and legacy versions rather than true duplicate errors. Overall, the dataset demonstrates strong structural quality and completeness, making it suitable for downstream exploratory analysis, recommendation modeling, and retention-focused investigation.

# Graphical Exploratory Data Analysis

In [ ]:
# explode genres
genres_exploded = df_clean.selectExpr('explode(genres_array) as genre')

# count frequency
genre_counts = (
    genres_exploded
    .groupBy('genre')
    .count()
    .orderBy('count', ascending=False)
    .limit(15)
)

# convert to pandas for plotting
genre_pd = genre_counts.toPandas().sort_values('count')

# plot
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 6))
plt.barh(genre_pd['genre'], genre_pd['count'])
plt.xlabel('Count')
plt.title('Top 15 Genres')
plt.show()

In [ ]:
# explode tags (map → key/value)
tags_exploded = df_clean.selectExpr('explode(tags_map) as (tag, weight)')

# count frequency (NOT weight)
tag_counts = (
    tags_exploded
    .groupBy('tag')
    .count()
    .orderBy('count', ascending=False)
    .limit(30)
)

# convert to pandas
tag_pd = tag_counts.toPandas().sort_values('count')

# plot
plt.figure(figsize=(16,8))
plt.tight_layout()
plt.barh(tag_pd['tag'], tag_pd['count'])
plt.xlabel('Count')
plt.yticks(fontsize=12)
plt.title('Top 30 Tags')
plt.show()

## EDA Insights — Genre & Tag Distribution

### Genre Distribution
- The dataset is heavily skewed toward **Indie**, which significantly exceeds all other genres, indicating a strong representation of smaller or independently developed titles.
- **Casual, Action, and Adventure** form a second tier of highly represented genres, suggesting a focus on broadly accessible and popular gameplay styles.
- Mid-tier genres such as **Simulation, Strategy, and RPG** show moderate representation, indicating diversity but less dominance.
- Lower-frequency genres (e.g., Sports, Racing, Utilities) are minimally represented, suggesting potential undercoverage of niche or specialized categories.

---

### Tag Distribution
- The most dominant tags—**Indie, Singleplayer, Casual, Adventure, and Action**—indicate that the dataset is strongly oriented toward **accessible, broad-appeal gaming experiences**.
- There is a noticeable emphasis on **singleplayer gameplay**, which contrasts with the competitive multiplayer focus seen in specific case studies like CS2 and Dota 2.
- Tags such as **2D, Simulation, Strategy, and Puzzle** highlight a significant presence of mechanically simpler or design-focused games.
- A long tail of tags (e.g., Horror, Relaxing, Family Friendly, Anime) reflects **diversity in themes and player experiences**, though each appears less frequently.

---

### Key Takeaway(s)
- The dataset exhibits a **clear skew toward Indie and Casual game design**, which may bias analyses toward simpler, accessible gameplay patterns.
- Competitive and multiplayer-heavy experiences exist but are **not the dominant representation**, suggesting that high-profile games (e.g., CS2, Dota 2) may not reflect the overall dataset distribution.

## Co-Occurrence Heatmap

In [ ]:
# explode genres
genres = df_clean.selectExpr('appid', 'explode(genres_array) as genre')

# self-join to get pairs
genre_pairs = genres.alias('a').join(
    genres.alias('b'),
    on='appid'
).select(
    F.col('a.genre').alias('genre_1'),
    F.col('b.genre').alias('genre_2')
)

# count pairs
pair_counts = (
    genre_pairs
    .groupBy('genre_1', 'genre_2')
    .count()
)

# convert to pandas
pair_pd = pair_counts.toPandas()

# pivot for heatmap
pivot = pair_pd.pivot(index='genre_1', columns='genre_2', values='count').fillna(0)

# plot
plt.figure(figsize=(12, 10))
sns.heatmap(pivot, cmap='Reds')
plt.title('Genre Co-Occurrence Heatmap')
plt.show()

In [ ]:
# get top 20 tags first
top_tags = (
    df_clean
    .selectExpr('explode(tags_map) as (tag, weight)')
    .groupBy('tag')
    .count()
    .orderBy('count', ascending=False)
    .limit(20)
    .select('tag')
)

# filter dataset to only top tags
tags = df_clean.selectExpr('appid', 'explode(tags_map) as (tag, weight)') \
    .join(top_tags, on='tag')

# self-join
tag_pairs = tags.alias('a').join(
    tags.alias('b'),
    on='appid'
).select(
    F.col('a.tag').alias('tag_1'),
    F.col('b.tag').alias('tag_2')
)

# count
tag_counts = tag_pairs.groupBy('tag_1', 'tag_2').count()

# pandas + pivot
tag_pd = tag_counts.toPandas()
pivot_tags = tag_pd.pivot(index='tag_1', columns='tag_2', values='count').fillna(0)

# plot
plt.figure(figsize=(12, 10))
sns.heatmap(pivot_tags, cmap='Greens')
plt.title('Top Tag Co-Occurrence Heatmap')
plt.show()

## Heatmap Insights

### Genre Co-Occurrence Heatmap
- The heatmap reveals a strong central cluster around **Indie, Action, Adventure, and Casual**, indicating these genres frequently appear together and form the **core of the dataset’s game ecosystem**.
- **Indie** shows consistently high co-occurrence across many genres, reinforcing its role as a **broad, cross-cutting classification** rather than a standalone gameplay type.
- **Action and Adventure** exhibit strong mutual relationships, suggesting they function as **complementary gameplay foundations** in many titles.
- More specialized genres such as **Strategy and Simulation** display more localized relationships, indicating **tighter, mechanic-driven groupings**.
- Several categories (e.g., Video Production, Education, Utilities) appear largely isolated, reflecting **non-game or niche classifications** within the dataset.

---

###  Tag Co-Occurrence Heatmap
- A clear cluster emerges around **Singleplayer, Indie, Casual, Adventure, and Action**, highlighting a dominant **accessible, broad-appeal gameplay segment**.
- Tags related to player experience, such as **Story Rich, Exploration, and Atmospheric**, tend to co-occur, suggesting a **narrative-focused gameplay grouping**.
- Mechanical and structural tags such as **Simulation, Strategy, and RPG** show moderate clustering, indicating **shared systems-based gameplay characteristics**.
- Visual and stylistic tags (e.g., **2D, Pixel Graphics, Cute**) form smaller clusters, reflecting **aesthetic-driven design patterns**.
- Compared to genres, tag relationships are more granular and reveal **distinct player experience archetypes**, rather than broad categorical groupings.

---

### Key Takeaway(s)
- Both heatmaps demonstrate that games are not isolated by a single category; instead, they exist within **interconnected clusters of genres and gameplay traits**.
- **Genres provide high-level structure**, while **tags reveal finer-grained player experiences**, together forming a layered view of the game ecosystem.

# Misc. Questions and Ideas to consider:

In [ ]:
# Which genres keep players playing longer?

genre_playtime = (
    df_clean
    .selectExpr('explode(genres_array) as genre', 'average_playtime_forever')
    .groupBy('genre')
    .avg('average_playtime_forever')
    .orderBy('avg(average_playtime_forever)', ascending=False)
    .limit(10)
    .toPandas()
)

plt.figure(figsize=(10,6))
plt.barh(genre_playtime['genre'], genre_playtime['avg(average_playtime_forever)'])
plt.title('Average Playtime by Genre')
plt.xlabel('Avg Playtime (minutes)')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# What gameplay features drive engagement?

tag_playtime = (
    df_clean
    .selectExpr('explode(tags_map) as (tag, weight)', 'average_playtime_forever')
    .groupBy('tag')
    .avg('average_playtime_forever')
    .withColumnRenamed('avg(average_playtime_forever)', 'avg_playtime')
    .orderBy('avg_playtime', ascending=False)
    .limit(15)
    .toPandas()
)

plt.figure(figsize=(10,6))
plt.barh(tag_playtime['tag'], tag_playtime['avg_playtime'])
plt.title('Average Playtime by Tag')
plt.xlabel('Avg Playtime')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Do highly played games get better reception?
pd_df = df_clean.select(
    'average_playtime_forever',
    'pct_pos_total',
    'num_reviews_total'
).toPandas()

plt.figure(figsize=(6,5))
plt.scatter(pd_df['average_playtime_forever'], pd_df['pct_pos_total'], alpha=0.3)
plt.xlabel('Avg Playtime')
plt.ylabel('Pct Positive')
plt.title('Engagement vs Sentiment')
plt.show()

In [ ]:
# Are popular games actually deeply played?
plt.figure(figsize=(6,5))
plt.scatter(pd_df['num_reviews_total'], pd_df['average_playtime_forever'], alpha=0.3)
plt.xlabel('Total Reviews (proxy for popularity)')
plt.ylabel('Avg Playtime')
plt.title('Popularity vs Engagement')
plt.show()

In [ ]:
# games released by year
release_year_counts = (
    df_clean
    .filter(F.col('release_dt').isNotNull())
    .withColumn('release_year', F.year(F.col('release_dt')))
    .groupBy('release_year')
    .count()
    .orderBy('release_year')
)

release_year_pd = release_year_counts.toPandas()

plt.figure(figsize=(14, 6))
plt.plot(release_year_pd['release_year'], release_year_pd['count'], marker='o')
plt.xlabel('Release Year')
plt.ylabel('Number of Games')
plt.title('Distribution of Game Releases Over Time')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## Engagement & Retention Insights

### Average Playtime by Genre

- Certain niche genres such as **Gore** and **Violent** appear with very high average playtime, suggesting these categories may attract **smaller but highly engaged player communities**.
- Non-traditional categories (e.g., **Web Publishing**, **Audio Production**) also rank highly, indicating possible **metadata noise or non-game software entries** within the dataset.
- More common genres such as **Simulation** and **Massively Multiplayer** appear lower in average playtime, suggesting that **broad popularity does not always translate into deeper long-term engagement**.
- Genre alone may not fully explain retention, since some of the highest playtime categories reflect niche use cases rather than dominant market segments.

---

### Average Playtime by Tag

- Several top tags (e.g., **Mature**, **Nudity**, **Sexual Content**) are associated with elevated playtime, suggesting that **specific niche content segments may generate prolonged engagement**.
- Tags such as **MMORPG** and **Dating Sim** also rank highly, indicating that **long-session or progression-based gameplay loops contribute strongly to retention**.
- Some tags (e.g., **Illuminati**, **Crowdfunded**) may reflect thematic or metadata labeling rather than gameplay mechanics, introducing noise.
- Compared with genres, tags provide a more granular lens into player behavior, suggesting that **engagement is often driven by specific motivations rather than broad categories alone**.

---

### Engagement vs Sentiment

- The plot shows a dense concentration of games at relatively low playtime levels, indicating that most titles do not sustain deep long-term engagement.
- High-playtime games appear across a wide range of review scores, suggesting that **strong engagement does not automatically imply stronger sentiment**.
- There is no clear linear relationship between playtime and positive review percentage.
- This implies that players may invest significant time in games for competition, habit, or progression even when overall satisfaction is mixed.

---

### Popularity vs Engagement

- Most titles cluster at the lower end of both review count and playtime, indicating a strong **long-tail market structure** where many games remain niche.
- A relatively small number of outliers account for extremely high engagement or review volume, showing that **a handful of dominant games capture disproportionate player attention**.
- Popularity and engagement are related but distinct; widely reviewed games are not always the most deeply played.
- This reinforces that visibility, ownership, and retention should be evaluated separately.

---

### Release Distribution Over Time

- The dataset contains relatively few titles before the early 2010s, followed by rapid growth beginning around 2014.
- This likely reflects expansion of the Steam marketplace, lower barriers for indie developers, and the continued rise of digital distribution.
- The sharp spike around **2023–2024** may represent post-pandemic development pipelines reaching completion alongside continued catalog expansion.
- The decline in the most recent year likely reflects incomplete current-year data rather than a true collapse in releases.

---

### Key Takeaway(s)

- Retention is not evenly distributed; it is often concentrated in **niche genres and highly specific gameplay experiences**.
- **Tags often reveal more about engagement than genres**, because they better capture player motivations and mechanics.
- **Popularity does not guarantee retention**, reinforcing the need to separate awareness from sustained play behavior.
- The platform exhibits a strong **long-tail ecosystem**, where a few titles dominate attention while many compete for smaller audiences.
- Modern releases enter a significantly more crowded marketplace than older titles, increasing substitution pressure and competition for player time.